# Bài tập Buổi 4 — Perceptron từ C / I / T

Notebook sinh viên đi song song với hai file Python:

- `bai-tap/buoi4_perceptron_cit.py`: một node, câu hỏi **có phải C không?**
- `bai-tap/buoi4_perceptron_cit_nhieu_dau_ra.py`: ba node song song cho C / I / T

Quy trình: **Predict → Code → Check → Explain**. Hãy ghi dự đoán trên giấy trước khi chạy ô kiểm tra.

## Mục tiêu và cách nộp

Sau bài này, bạn cần tính đúng `net`, `ŷ`, `e`, cập nhật `w/b`, giải thích vì sao chỉ feature đang bật thay đổi, và vẽ được ba Perceptron cùng nhận một vector input.

1. Hoàn thiện các TODO trong hai file Python.
2. Chạy hết các ô có `run_checks()`.
3. Điền phần giải thích ngắn ở cuối notebook và lưu notebook cùng code.

> Perceptron là một cách xây dựng **node nơ-ron nhân tạo cơ bản**. Nhiều node cùng lớp và nhiều lớp nối tiếp nhau tạo thành mạng nơ-ron lớn hơn.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'bai-tap').exists():
    ROOT = ROOT.parent

def load_module(name, relative_path):
    path = ROOT / relative_path
    assert path.exists(), f'Không tìm thấy {path}'
    spec = spec_from_file_location(name, path)
    module = module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

starter = load_module('buoi4_perceptron_cit', Path('bai-tap/buoi4_perceptron_cit.py'))
multi = load_module('buoi4_perceptron_cit_nhieu_dau_ra', Path('bai-tap/buoi4_perceptron_cit_nhieu_dau_ra.py'))
print('Đã nạp starter một đầu ra và bài mở rộng nhiều đầu ra.')

## A. Dự đoán trước khi chạy — một Perceptron nhận biết C

Dữ liệu là `x_C=(1,0,1,1,0)`, `x_I=(0,0,0,0,1)`, `x_T=(0,0,1,0,1)`, `η=0.5`, `w⁽⁰⁾=0`, `b⁽⁰⁾=-0.25`. Điền bảng bằng tay trước.

In [ ]:
# TODO: điền các giá trị dự đoán của bạn; chưa cần gọi code ở đây.
du_doan_bang_tay = {
    'C': {'net': None, 'y_hat': None, 'cap_nhat': None},
    'I': {'net': None, 'y_hat': None, 'cap_nhat': None},
    'T': {'net': None, 'y_hat': None, 'cap_nhat': None},
}
du_doan_bang_tay

## B. Hoàn thiện starter một đầu ra

Mở `bai-tap/buoi4_perceptron_cit.py` và hoàn thiện TODO 1–3. Công thức cần dùng:

```text
net = w · x + b
ŷ = 1 nếu net >= 0, ngược lại 0
e = y - ŷ
w_mới = w + η e x;  b_mới = b + η e
```

In [ ]:
# Ô này chỉ vượt qua sau khi bạn hoàn thiện starter.
starter.run_checks()
print('✓ Starter một đầu ra đã vượt qua kiểm tra.')

In [ ]:
# Đối chiếu log ba lượt sau khi starter đã chạy được.
w_final, b_final, history = starter.train_one_epoch()
for row in history:
    print(row['mau'], 'net=', row['net'], 'ŷ=', row['y_hat'], 'e=', row['error'],
          'w=', row['w_sau_luot'], 'b=', row['b_sau_luot'])

## C. Mẫu C bị thiếu một nét

Tạo một biến thể của C, ví dụ tắt feature nét ngang phía trên: `x_C_thieu=(1,0,0,1,0)`. Dùng `w_final, b_final` của một epoch để tính `net` và giải thích vì sao kết quả có thể thay đổi.

In [ ]:
x_C_thieu = starter.X[0].copy()
x_C_thieu[2] = 0  # TODO: thử thêm một biến thể khác
net_C_thieu, yhat_C_thieu = starter.predict(x_C_thieu, w_final, b_final)
print('x biến thể:', x_C_thieu)
print('net =', net_C_thieu, 'ŷ =', yhat_C_thieu)
# TODO: viết 2–3 câu giải thích vào ô Markdown kế tiếp.

**Giải thích của tôi:**

- Feature nào đang là bằng chứng mạnh nhất cho C?
- Bias làm thay đổi quyết định như thế nào?
- Nếu chỉ lật một pixel trong lưới 4×4, feature nào có thể đổi?

## D. Mở rộng: ba Perceptron song song

Mỗi node nhận cùng `x` nhưng có hàng trọng số riêng:

```text
x ──► P_C: W_C·x+b_C ──► score_C ┐
x ──► P_I: W_I·x+b_I ──► score_I ├─► argmax ─► lớp C/I/T
x ──► P_T: W_T·x+b_T ──► score_T ┘
```

Hoàn thiện TODO 1–4 trong `buoi4_perceptron_cit_nhieu_dau_ra.py`. Mục tiêu one-vs-rest là C→`[1,0,0]`, I→`[0,1,0]`, T→`[0,0,1]`.

In [ ]:
# Ô này kiểm tra cả một epoch và quá trình hội tụ nhiều epoch.
multi.run_checks()
W, b, mistakes = multi.fit()
print('Số mẫu sai theo epoch:', mistakes)
print('Dự đoán cuối:', [multi.predict_class(x, W, b) for x in multi.X])

## E. Phân biệt hai chữ `net`

Viết ngắn gọn 4–6 dòng: `net = wᵀx + b` của Perceptron nhận feature và phân loại **một mẫu**; còn `netᵢ = Σⱼ wᵢⱼxⱼ` của Hopfield là tổng tín hiệu để cập nhật **một pixel/neuron trong pattern**. Hai công thức cùng là tổng có trọng số nhưng khác input, bias, output và mục đích.

## F. Challenge XOR và checklist nộp bài

Một Perceptron đơn chỉ tạo được một đường biên tuyến tính, nên không tách được bốn điểm XOR. Hãy giải thích điều này bằng lời và nêu hidden layer của MLP giúp gì.

- [ ] Starter một đầu ra vượt qua `run_checks()`.
- [ ] Bảng tay C/I/T có `net`, `ŷ`, `e`, `w`, `b`.
- [ ] Có một mẫu C bị thiếu nét và phần giải thích.
- [ ] File nhiều đầu ra vượt qua `run_checks()` và dự đoán đúng C/I/T.
- [ ] Đã viết phân biệt `net` Perceptron / Hopfield và giới hạn XOR.